# PowerOps_v1 — Notebook 02: Document Preparation

This notebook converts the validated, normalized DevOps issues from Notebook
01 (`data/normalized_issues.json`) into LangChain `Document` objects — the
unit that gets embedded and stored in Pinecone in Notebook 03.

**Two things every Document needs, and why they're split apart:**

1. **`page_content`** — the text that gets embedded and semantically searched.
   This is what lets a question like *"certificate expiry problems"* find an
   issue whose Summary says *"Cert IDPPNCID2 - Certificate for NCID site in
   Pre-Prod  Expires: 03/27/2027"* even though the words don't match exactly.
2. **`metadata`** — structured fields (team, assignee, priority, status,
   dates) that Pinecone can filter on *exactly*. This is what lets a question
   like *"critical issues for Nova Team"* be answered by a hard filter
   instead of hoping semantic similarity happens to rank the right team's
   issues higher.

### A note on this dataset specifically

Recall from Notebook 01: the source data has **no separate description,
comments, or resolution fields** — `summary` is the only free text. So
`page_content` here is built from `summary` plus the metadata fields rendered
as readable text (confirmed as the right approach for this dataset). This
also means each issue's content is short, which matters for the chunking
discussion below.


## 1. Imports and load validated data

In [1]:
import json
from pathlib import Path

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

NORMALIZED_PATH = Path("../data/normalized_issues.json")

with open(NORMALIZED_PATH, "r", encoding="utf-8") as f:
    issues = json.load(f)

print(f"Loaded {len(issues)} validated issues from {NORMALIZED_PATH.name}")
issues[0]


Loaded 1000 validated issues from normalized_issues.json


{'issue_key': 'INO-21920',
 'issue_id': '7798481',
 'summary': 'SDX case  are creating but remaining in Delayed Processing Pending status -FT13',
 'assignee': 'Sullivan, Deepa (Contractor)',
 'assignee_id': '2ca25fab6856f67786767b43',
 'reporter': 'Sullivan, Deepa (Contractor)',
 'reporter_id': '2ca25fab6856f67786767b43',
 'status': 'In Progress',
 'priority': 'Medium',
 'assigned_team': 'Falcon Squad',
 'story_points': 0.25,
 'updated_raw': '5/18/26 12:01',
 'updated_dt': '2026-05-18T12:01:00',
 'due_date_raw': '5/15/26 0:00',
 'due_date_dt': '2026-05-15T00:00:00'}

## 2. Render readable `page_content`

Each issue becomes a short, human-readable text block. Keeping the format
consistent (labeled fields, one per line) helps the embedding model capture
both the semantic meaning *and* the structured facts, even though the
structured facts are also stored separately as metadata.

Fields with no value (e.g. `due_date_raw` when it's blank, or `story_points`
when it's `None`) are rendered as `Not set` rather than omitted, so the LLM
never has to guess whether a field was left out of the export vs. genuinely
absent for that issue.


In [2]:
def render_page_content(issue: dict) -> str:
    """Render one normalized issue as a readable text block for embedding."""
    story_points = issue.get("story_points")
    story_points_str = str(story_points) if story_points is not None else "Not set"
    due_date = issue.get("due_date_raw") or "Not set"

    return (
        f"Issue Key: {issue['issue_key']}\n"
        f"Summary: {issue['summary']}\n"
        f"Assigned Team: {issue['assigned_team']}\n"
        f"Assignee: {issue['assignee']}\n"
        f"Reporter: {issue.get('reporter') or 'Not set'}\n"
        f"Priority: {issue['priority']}\n"
        f"Status: {issue['status']}\n"
        f"Story Points: {story_points_str}\n"
        f"Last Updated: {issue.get('updated_raw') or 'Not set'}\n"
        f"Due Date: {due_date}"
    )


print(render_page_content(issues[0]))


Issue Key: INO-21920
Summary: SDX case  are creating but remaining in Delayed Processing Pending status -FT13
Assigned Team: Falcon Squad
Assignee: Sullivan, Deepa (Contractor)
Reporter: Sullivan, Deepa (Contractor)
Priority: Medium
Status: In Progress
Story Points: 0.25
Last Updated: 5/18/26 12:01
Due Date: 5/15/26 0:00


## 3. Build metadata

Metadata is what Pinecone's filter engine operates on, so every field a
question might filter by needs to be here — and needs to be a Pinecone-safe
type (string, number, boolean, or list of strings; not `None`, not a Python
`datetime`).

We keep `updated_dt`/`due_date_dt` as **Unix timestamps (float)** rather than
datetime objects or ISO strings — Pinecone metadata filters support numeric
range queries (`$gte`/`$lte`), which is exactly what a question like *"issues
updated in the last week"* would need later. A missing due date becomes `0.0`
with a paired `has_due_date: False` flag, so "no due date" is distinguishable
from "due date of the epoch."


In [3]:
from datetime import datetime


def _to_epoch(dt_str: str | None) -> float | None:
    """Convert an ISO datetime string (from Notebook 01) to a Unix timestamp."""
    if not dt_str:
        return None
    try:
        return datetime.fromisoformat(dt_str).timestamp()
    except ValueError:
        return None


def build_metadata(issue: dict) -> dict:
    """Build Pinecone-safe metadata for one issue."""
    updated_epoch = _to_epoch(issue.get("updated_dt"))
    due_epoch = _to_epoch(issue.get("due_date_dt"))

    return {
        "issue_key": issue["issue_key"],
        "issue_id": issue["issue_id"],
        "assigned_team": issue["assigned_team"],
        "assignee": issue["assignee"],
        "reporter": issue.get("reporter") or "",
        "priority": issue["priority"],
        "status": issue["status"],
        "story_points": float(issue["story_points"]) if issue.get("story_points") is not None else -1.0,
        "has_story_points": issue.get("story_points") is not None,
        "updated_raw": issue.get("updated_raw") or "",
        "updated_epoch": updated_epoch if updated_epoch is not None else 0.0,
        "due_date_raw": issue.get("due_date_raw") or "",
        "due_date_epoch": due_epoch if due_epoch is not None else 0.0,
        "has_due_date": due_epoch is not None,
    }


build_metadata(issues[0])


{'issue_key': 'INO-21920',
 'issue_id': '7798481',
 'assigned_team': 'Falcon Squad',
 'assignee': 'Sullivan, Deepa (Contractor)',
 'reporter': 'Sullivan, Deepa (Contractor)',
 'priority': 'Medium',
 'status': 'In Progress',
 'story_points': 0.25,
 'has_story_points': True,
 'updated_raw': '5/18/26 12:01',
 'updated_epoch': 1779120060.0,
 'due_date_raw': '5/15/26 0:00',
 'due_date_epoch': 1778817600.0,
 'has_due_date': True}

## 4. Build one Document per issue

In [4]:
def build_document(issue: dict) -> Document:
    """Build a single LangChain Document for one normalized DevOps issue."""
    return Document(
        page_content=render_page_content(issue),
        metadata=build_metadata(issue),
    )


documents = [build_document(issue) for issue in issues]
print(f"Built {len(documents)} Documents.")
print("-" * 60)
print(documents[0].page_content)
print("-" * 60)
print(documents[0].metadata)


Built 1000 Documents.
------------------------------------------------------------
Issue Key: INO-21920
Summary: SDX case  are creating but remaining in Delayed Processing Pending status -FT13
Assigned Team: Falcon Squad
Assignee: Sullivan, Deepa (Contractor)
Reporter: Sullivan, Deepa (Contractor)
Priority: Medium
Status: In Progress
Story Points: 0.25
Last Updated: 5/18/26 12:01
Due Date: 5/15/26 0:00
------------------------------------------------------------
{'issue_key': 'INO-21920', 'issue_id': '7798481', 'assigned_team': 'Falcon Squad', 'assignee': 'Sullivan, Deepa (Contractor)', 'reporter': 'Sullivan, Deepa (Contractor)', 'priority': 'Medium', 'status': 'In Progress', 'story_points': 0.25, 'has_story_points': True, 'updated_raw': '5/18/26 12:01', 'updated_epoch': 1779120060.0, 'due_date_raw': '5/15/26 0:00', 'due_date_epoch': 1778817600.0, 'has_due_date': True}


## 5. One document per issue, or multiple chunks?

Let's check empirically rather than guessing. We measure the character length
of every `page_content` we just built, and compare against a conservative
chunk-size threshold.

A typical embedding model (e.g. OpenAI `text-embedding-3-small`) comfortably
handles a few thousand tokens (~4 chars/token as a rough rule of thumb), so
anything under roughly 1500 characters is nowhere near a concern. Given this
dataset's `page_content` is Summary + a handful of metadata lines (no long
description/comments), we expect every document to be far under that.


In [5]:
lengths = [len(doc.page_content) for doc in documents]

print(f"Documents: {len(lengths)}")
print(f"Min length:    {min(lengths)} chars")
print(f"Max length:    {max(lengths)} chars")
print(f"Mean length:   {sum(lengths) / len(lengths):.1f} chars")
print(f"95th pct:      {sorted(lengths)[int(len(lengths) * 0.95)]} chars")

CHUNK_THRESHOLD_CHARS = 1500
over_threshold = [l for l in lengths if l > CHUNK_THRESHOLD_CHARS]
print(f"\nDocuments over {CHUNK_THRESHOLD_CHARS} chars (would need chunking): {len(over_threshold)}")


Documents: 1000
Min length:    216 chars
Max length:    393 chars
Mean length:   279.2 chars
95th pct:      327 chars

Documents over 1500 chars (would need chunking): 0


## 6. Optional chunking (future-proofing)

As confirmed above, no document in this dataset actually needs chunking. But
PowerOps should not silently mishandle a future issue with a much longer
Summary or an added description field, so we implement chunking as an
**opt-in, threshold-gated** step using `RecursiveCharacterTextSplitter`:

- Documents at or under the threshold pass through unchanged, with a single
  `chunk_id = 0` added to their metadata (so ingestion always deals with
  "documents," never a mix of chunked/unchunked shapes).
- Documents over the threshold get split, and **every resulting chunk
  inherits the parent issue's full metadata** (`issue_key`, `assigned_team`,
  `assignee`, `priority`, `status`, ...) plus its own `chunk_id`. This is
  critical — a chunk that lost its `assigned_team` metadata would become
  unfindable by any team-filtered query, and a citation without `issue_key`
  would be useless as evidence.


In [6]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_THRESHOLD_CHARS,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""],
)


def chunk_document(doc: Document) -> list[Document]:
    """Split a Document into chunks if it exceeds the threshold; otherwise
    return it unchanged (wrapped as a single 'chunk 0').
    """
    if len(doc.page_content) <= CHUNK_THRESHOLD_CHARS:
        chunk_meta = dict(doc.metadata)
        chunk_meta["chunk_id"] = 0
        chunk_meta["chunk_count"] = 1
        return [Document(page_content=doc.page_content, metadata=chunk_meta)]

    pieces = text_splitter.split_text(doc.page_content)
    chunks = []
    for i, piece in enumerate(pieces):
        chunk_meta = dict(doc.metadata)
        chunk_meta["chunk_id"] = i
        chunk_meta["chunk_count"] = len(pieces)
        chunks.append(Document(page_content=piece, metadata=chunk_meta))
    return chunks


all_chunks = []
for doc in documents:
    all_chunks.extend(chunk_document(doc))

print(f"Documents in:  {len(documents)}")
print(f"Chunks out:    {len(all_chunks)}")
print(f"Docs that were actually split into >1 chunk: "
      f"{sum(1 for d in documents if len(d.page_content) > CHUNK_THRESHOLD_CHARS)}")


Documents in:  1000
Chunks out:    1000
Docs that were actually split into >1 chunk: 0


## 7. Vector IDs

Notebook 03 needs a stable, unique ID per vector for idempotent upserts
(re-running ingestion should update existing vectors, not duplicate them). We
use `{issue_key}::chunk{chunk_id}` — stable across re-runs since it's derived
purely from data, not from insertion order.


In [7]:
def vector_id_for(doc: Document) -> str:
    """Build a stable Pinecone vector ID for a chunk."""
    return f"{doc.metadata['issue_key']}::chunk{doc.metadata['chunk_id']}"


for doc in all_chunks[:3]:
    print(vector_id_for(doc))


INO-21920::chunk0
INO-21914::chunk0
INO-21903::chunk0


## 8. Print several resulting documents and their metadata

In [8]:
import random

random.seed(42)
sample = random.sample(all_chunks, 5)

for doc in sample:
    print("=" * 70)
    print(f"Vector ID: {vector_id_for(doc)}")
    print("-" * 70)
    print(doc.page_content)
    print("-" * 70)
    print("Metadata:")
    for k, v in doc.metadata.items():
        print(f"  {k}: {v}")


Vector ID: INO-20302::chunk0
----------------------------------------------------------------------
Issue Key: INO-20302
Summary: Unlock account
Assigned Team: Nova Team
Assignee: Shaw, Elizabeth
Reporter: Simmons, Anna
Priority: Medium
Status: Done
Story Points: 1.0
Last Updated: 4/13/26 16:31
Due Date: 4/13/26 0:00
----------------------------------------------------------------------
Metadata:
  issue_key: INO-20302
  issue_id: 7783719
  assigned_team: Nova Team
  assignee: Shaw, Elizabeth
  reporter: Simmons, Anna
  priority: Medium
  status: Done
  story_points: 1.0
  has_story_points: True
  updated_raw: 4/13/26 16:31
  updated_epoch: 1776112260.0
  due_date_raw: 4/13/26 0:00
  due_date_epoch: 1776052800.0
  has_due_date: True
  chunk_id: 0
  chunk_count: 1
Vector ID: INO-21386::chunk0
----------------------------------------------------------------------
Issue Key: INO-21386
Summary: PCP1 - disable weekend shutdown (5/1/2026  -  5/3/2026)
Assigned Team: Falcon Squad
Assignee: Ma

## Why metadata is critical for PowerOps queries

Semantic similarity alone cannot reliably answer questions like *"What
critical issues are assigned to Falcon Squad?"* — there is nothing
semantically special about the word "Falcon Squad" that would make the
embedding model rank Falcon Squad's issues above Nova Team's or Summit
Crew's for an otherwise-generic query about "critical issues." Team names,
assignee names, priority levels, and statuses are **categorical facts**, not
concepts with fuzzy semantic neighborhoods — they need **exact filtering**,
not similarity ranking.

This is exactly why every Document here carries both:

- `page_content` for the parts of a question that *are* genuinely semantic
  ("certificate problems", "Kubernetes issues", "networking failures"), and
- `metadata` for the parts that are structured constraints (team, assignee,
  priority, status, dates) — enforced with Pinecone's exact-match/range
  metadata filters in Notebook 04, *before* semantic ranking runs on whatever
  is left.

Skipping metadata (or losing it during chunking) would force the whole system
onto semantic-only retrieval, which is unreliable for exactly the
structured-filter questions PowerOps is meant to handle well.


## 9. Persist the prepared chunks for Notebook 03

We serialize `all_chunks` to a flat JSON list (`page_content` + `metadata` +
precomputed `vector_id`) so Notebook 03 can load it directly rather than
re-deriving document construction.


In [9]:
OUTPUT_PATH = Path("../data/prepared_documents.json")

serialized = [
    {
        "vector_id": vector_id_for(doc),
        "page_content": doc.page_content,
        "metadata": doc.metadata,
    }
    for doc in all_chunks
]

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(serialized, f, indent=2)

print(f"Saved {len(serialized)} prepared documents/chunks to {OUTPUT_PATH.resolve()}")


Saved 1000 prepared documents/chunks to /Users/svedamur/Documents/agentic-ai-simulations/devOps_rag_week2/data/prepared_documents.json


## Summary & next steps

- Built one `Document` per validated issue: `page_content` = rendered
  Summary + metadata text block; `metadata` = Pinecone-safe structured fields
  (`issue_key`, `assigned_team`, `assignee`, `priority`, `status`,
  `story_points`, `updated_epoch`, `due_date_epoch`, plus presence flags).
- Measured actual content lengths and confirmed **no chunking is currently
  needed** for this dataset — but implemented threshold-gated chunking with
  `RecursiveCharacterTextSplitter` so a future longer-text issue is handled
  safely, with full metadata inheritance per chunk.
- Defined a stable `vector_id` scheme (`{issue_key}::chunk{chunk_id}`) for
  idempotent Pinecone upserts.
- Saved **`data/prepared_documents.json`** — ready for embedding.

**Next: Notebook 03 — Pinecone Ingestion.** We'll load `.env` configuration,
initialize the embedding model and Pinecone client, create/connect to the
index, upsert these prepared documents idempotently, and run test similarity
searches.
